In [ ]:
# Ensure Internet toggle on the right side panel is switched ON!
!nvidia-smi

In [ ]:
%%bash
# 1. Clean up any corrupted directory versions cleanly
rm -rf /kaggle/working/spec-fastgs

# 2. Recursively clone using the correct branch name structure
git clone --recursive -b main https://github.com/0Nguyen0Cong0Tuan0/thesis-all.git /kaggle/working/spec-fastgs

# 3. Verify the layout structure
echo "📂 Verifying submodules directory contents:"
ls -la /kaggle/working/spec-fastgs/spec-fastgs/submodules/

In [ ]:
%%bash
# Copy datasets from the read-only input mount into the WRITABLE working dir.
# (The synthetic loaders write points3d.ply into the scene dir, so the source MUST
#  be writable — /kaggle/input is read-only.)
WORK=/kaggle/working/spec-fastgs/spec-fastgs/datasets
mkdir -p "$WORK"

# Mip-NeRF 360 — auto-locate under /kaggle/input, trying several strategies in order,
# since the exact folder name/casing/nesting depends on how the dataset was attached.
# CONFIRMED (2026-07-05, via screenshot of the actual Kaggle Input panel): the real
# layout is spec-fastgs-datasets/datasets/datasets/mipnerf360/{bicycle,bonsai,counter,
# ...} — i.e. mipnerf360 sits 6 levels below /kaggle/input (datasets/nctuan/
# spec-fastgs-datasets/datasets/datasets/mipnerf360), one "datasets/" deeper than the
# old hardcoded path assumed. maxdepth is generously padded past that confirmed depth.
MIPNERF_SRC=""
# Strategy 1: exact name, case-insensitive, generous depth.
MIPNERF_SRC=$(find /kaggle/input -maxdepth 10 -iname "mipnerf360" -type d 2>/dev/null | head -1)
# Strategy 2: anchor on the actual scene this run needs (counter/images), in case the
# parent folder is named/nested differently than we expect.
if [ -z "$MIPNERF_SRC" ]; then
    COUNTER_IMAGES=$(find /kaggle/input -maxdepth 12 -type d -ipath "*counter/images" 2>/dev/null | head -1)
    if [ -n "$COUNTER_IMAGES" ]; then
        MIPNERF_SRC=$(dirname "$(dirname "$COUNTER_IMAGES")")
    fi
fi

if [ -n "$MIPNERF_SRC" ]; then
    echo "📥 copying $MIPNERF_SRC -> $WORK/mipnerf360"
    cp -r "$MIPNERF_SRC" "$WORK/mipnerf360"
else
    echo "⚠️  mipnerf360 NOT found under /kaggle/input — dumping the actual input"
    echo "   layout below (up to 6 levels) so the path can be fixed by hand:"
    find /kaggle/input -maxdepth 6 | sort
fi

# Synthetic suites — auto-locate under /kaggle/input regardless of the dataset slug.
for d in Anisotropic-Synthetic-Dataset Synthetic_NSVF; do
    SRC=$(find /kaggle/input -maxdepth 6 -iname "$d" -type d 2>/dev/null | head -1)
    if [ -n "$SRC" ]; then
        echo "📥 copying $SRC -> $WORK/"
        cp -r "$SRC" "$WORK/"
    else
        echo "⚠️  $d NOT found under /kaggle/input"
    fi
done
echo "📂 datasets now in working:"; ls "$WORK"

In [ ]:
%%bash
# Clean existing conda toolchain directories safely
rm -rf /opt/conda

# Reinstall isolated Miniconda (Python 3.10)
wget -q https://repo.anaconda.com/miniconda/Miniconda3-py310_23.11.0-1-Linux-x86_64.sh
bash Miniconda3-py310_23.11.0-1-Linux-x86_64.sh -b -p /opt/conda

# Activate custom installation
source /opt/conda/bin/activate

# Install compiler dependencies and CUDA Toolkit 11.7 matching constraints
/opt/conda/bin/conda install -y -c conda-forge cudatoolkit-dev=11.7 gcc_linux-64=11 gxx_linux-64=11

# Export local paths
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH

echo "----- NVCC -----"
nvcc --version
echo "----- PTXAS -----"
ptxas --version

In [ ]:
%%bash
/opt/conda/bin/pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --index-url https://download.pytorch.org/whl/cu117

In [ ]:
%%bash
/opt/conda/bin/python - << 'EOF'
import torch
print("Torch version:", torch.__version__)
print("CUDA back-end:", torch.version.cuda)
print("GPU Available:", torch.cuda.is_available())
EOF

In [ ]:
%%bash
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib:$LD_LIBRARY_PATH
export CC=gcc-11
export CXX=g++-11
export CUDAHOSTCXX=g++-11

BASE_DIR="/kaggle/working/spec-fastgs/spec-fastgs/submodules"

# 1. diff-gaussian-rasterization
cd "$BASE_DIR/diff-gaussian-rasterization_fastgs"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

# 2. simple-knn
cd "../simple-knn"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

# 3. fused-ssim
cd "../fused-ssim"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

In [ ]:
%%bash
/opt/conda/bin/python - << 'EOF'
import diff_gaussian_rasterization_fastgs
import simple_knn
import fused_ssim
print("✅ FastGS CUDA extensions compiled and loaded successfully!")
EOF

In [ ]:
%%bash
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib:$LD_LIBRARY_PATH
export CC=gcc-11
export CXX=g++-11
export CUDAHOSTCXX=g++-11

BASE_DIR="/kaggle/working/spec-fastgs/spec-fastgs/submodules"

cd "$BASE_DIR/diff-gaussian-rasterization_fastgs"
/opt/conda/bin/python setup.py bdist_wheel

cd "../simple-knn"
/opt/conda/bin/python setup.py bdist_wheel

cd "../fused-ssim"
/opt/conda/bin/python setup.py bdist_wheel

In [ ]:
%%bash
SRC="/kaggle/working/spec-fastgs/spec-fastgs/submodules"
DEST="/kaggle/working/fastgs_wheels_py310"

mkdir -p "$DEST"
find "$SRC" -name "*.whl" -exec cp {} "$DEST" \;

echo "✨ Wheels safely compiled and extracted to: $DEST"
ls -la "$DEST"

In [ ]:
%%bash
/opt/conda/bin/pip uninstall -y numpy
/opt/conda/bin/pip install "numpy<2" plyfile websockets tqdm imageio

In [ ]:
%%bash
/opt/conda/bin/python -c "import fused_ssim; import diff_gaussian_rasterization_fastgs; import plyfile; print('🎉 All systems functional and ready for execution!')"

In [ ]:
%%bash
# ============================================================
# ASG_DEGREE SWEEP PREREQUISITE -- extract the Reflection Prior (Shafer/Klinker,
# extract_reflection_prior.py) ONCE for counter. USE_REF_SCORE=True is shared across
# all three ASG_DEGREE runs below and the prior only depends on the scene/images, not
# asg_degree, so extracting it once here (sequentially, on GPU 0) avoids two GPU
# processes racing to write the same
# datasets/mipnerf360/counter/reflection_prior/*.png files when the ASG 32/48 cell
# launches them in parallel. (run_spec-fastgs_big.sh's own extraction step is also
# idempotent and will skip re-running this if the directory is already populated.)
# ============================================================
set -e
export PATH=/opt/conda/bin:$PATH
source /opt/conda/bin/activate
export CUDA_HOME=/opt/conda
export LD_LIBRARY_PATH=/opt/conda/lib:$LD_LIBRARY_PATH
export CUDA_VISIBLE_DEVICES=0

cd /kaggle/working/spec-fastgs/spec-fastgs

# Dataset layout guard (idempotent -- same as the training cells)
if [ -d "./datasets/datasets" ]; then
    echo "Re-aligning dataset file structure..."
    mv ./datasets/datasets/* ./datasets/
    rm -rf ./datasets/datasets
fi
if [ ! -d "./datasets/mipnerf360/counter/images_8" ] && [ ! -d "./datasets/mipnerf360/counter/images" ]; then
    echo "❌ ./datasets/mipnerf360/counter/images(_8) not found."
    echo "   The dataset-copy cell above could not locate the mipnerf360 folder under"
    echo "   /kaggle/input. Fix the source path in that cell, re-run it, then re-run this cell."
    exit 1
fi
echo "✅ dataset ready"

python extract_reflection_prior.py \
    -s ./datasets/mipnerf360/counter \
    -i images_8 \
    --sk_intensity 0.7 \
    --sk_saturation 0.2

echo "reflection priors written:"
ls ./datasets/mipnerf360/counter/reflection_prior | head -5
echo "... total:" $(ls ./datasets/mipnerf360/counter/reflection_prior/*.png | wc -l) "png priors"


In [ ]:
%%bash
# ============================================================
# ASG_DEGREE SWEEP, PART 1/2 -- USE_REF_SCORE=True, counter (Mip-NeRF 360)
# Runs ASG_DEGREE=32 on GPU 0 and ASG_DEGREE=48 on GPU 1 AT THE SAME TIME (Kaggle's
# T4 x2 gives two independent physical GPUs -- CUDA_VISIBLE_DEVICES=0 / =1 restricts
# each subprocess to see only its assigned one, so they don't contend for the same
# device). Both write to their own output dir
# (./output/counter_asg32_ref, ./output/counter_asg48_ref) so they can't clobber each
# other. `wait` blocks this cell until BOTH finish -- Jupyter's own cell-completion
# signal only fires after that, so the archive cell below is safe to run right after.
# ============================================================
# NOTE: no "set -e" here on purpose -- if either backgrounded job fails,
# "wait $PID" returns that job's exit status, and set -e would abort the
# script AT THAT wait CALL, before the tail/status diagnostics below ever run.
export PATH=/opt/conda/bin:$PATH
source /opt/conda/bin/activate
export CUDA_HOME=/opt/conda
export LD_LIBRARY_PATH=/opt/conda/lib:$LD_LIBRARY_PATH

cd /kaggle/working/spec-fastgs/spec-fastgs

if [ -d "./datasets/datasets" ]; then
    mv ./datasets/datasets/* ./datasets/
    rm -rf ./datasets/datasets
fi

echo "=== launching ASG_DEGREE=32 on GPU 0 ==="
CUDA_VISIBLE_DEVICES=0 SCENE=counter IMAGES=images_8 ASG_DEGREE=32 USE_REF_SCORE=True \
    OUTPUT_SUFFIX=_asg32_ref \
    bash run_spec-fastgs_big.sh > /kaggle/working/asg32_ref.log 2>&1 &
PID32=$!

echo "=== launching ASG_DEGREE=48 on GPU 1 ==="
CUDA_VISIBLE_DEVICES=1 SCENE=counter IMAGES=images_8 ASG_DEGREE=48 USE_REF_SCORE=True \
    OUTPUT_SUFFIX=_asg48_ref \
    bash run_spec-fastgs_big.sh > /kaggle/working/asg48_ref.log 2>&1 &
PID48=$!

echo "both launched (pid32=$PID32, pid48=$PID48) -- waiting for both to finish..."
wait $PID32
STATUS32=$?
wait $PID48
STATUS48=$?

echo "--- tail of asg32_ref.log ---"; tail -n 40 /kaggle/working/asg32_ref.log
echo "--- tail of asg48_ref.log ---"; tail -n 40 /kaggle/working/asg48_ref.log

echo "ASG_DEGREE=32 exit status: $STATUS32"
echo "ASG_DEGREE=48 exit status: $STATUS48"
if [ $STATUS32 -ne 0 ] || [ $STATUS48 -ne 0 ]; then
    echo "⚠️  at least one run failed -- check the logs above / full logs at"
    echo "   /kaggle/working/asg32_ref.log and /kaggle/working/asg48_ref.log"
    exit 1
fi


In [ ]:
%%bash
# ============================================================
# ASG_DEGREE SWEEP, PART 2/2 -- USE_REF_SCORE=True, counter (Mip-NeRF 360)
# ASG_DEGREE=64, run alone now that both GPUs are free (a 3rd run could also be
# started on GPU 1 in parallel with a re-run of one of the first two, but there's
# nothing left to pair it with here since both prior configs already completed).
# ============================================================
set -e
export PATH=/opt/conda/bin:$PATH
source /opt/conda/bin/activate
export CUDA_HOME=/opt/conda
export LD_LIBRARY_PATH=/opt/conda/lib:$LD_LIBRARY_PATH

cd /kaggle/working/spec-fastgs/spec-fastgs

CUDA_VISIBLE_DEVICES=0 SCENE=counter IMAGES=images_8 ASG_DEGREE=64 USE_REF_SCORE=True \
    OUTPUT_SUFFIX=_asg64_ref \
    bash run_spec-fastgs_big.sh


In [ ]:
import shutil, os

# One archive per ASG_DEGREE config -- keeps each zip small/independent, and a
# failed run in one config still lets you collect whichever others succeeded.
runs = [
    ("counter_asg32_ref", "spec_fastgs_output_asg32_ref"),
    ("counter_asg48_ref", "spec_fastgs_output_asg48_ref"),
    ("counter_asg64_ref", "spec_fastgs_output_asg64_ref"),
]

for scene_dir, out_name in runs:
    src = f"/kaggle/working/spec-fastgs/spec-fastgs/output/{scene_dir}"
    out = f"/kaggle/working/{out_name}"
    if os.path.isdir(src):
        shutil.make_archive(out, "zip", src)
        print("archived:", out + ".zip", round(os.path.getsize(out + ".zip") / 1e6, 1), "MB")
    else:
        print(f"no {scene_dir} output found at", src)
